# Domain-Specific RAG: Baseline Vector vs BM25+Vector Hybrid
**Domain:** Nginx admin knowledge base (60 synthetic docs, 60 test questions)
**Goal:** Compare plain dense-vector retrieval against a BM25+vector hybrid on retrieval accuracy (Recall@k, MRR).

Upload `docs.jsonl` and `qa_test.jsonl` to `/content/data/` before running (or mount Drive)."


In [ ]:
!pip install -q sentence-transformers faiss-cpu rank_bm25

In [ ]:
import json, numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

docs = [json.loads(l) for l in open('/content/data/docs.jsonl')]
qa   = [json.loads(l) for l in open('/content/data/qa_test.jsonl')]
texts = [d['text'] for d in docs]
ids   = [d['id'] for d in docs]

print(f"{len(docs)} docs, {len(qa)} test questions")

## Step 1: Baseline — dense vector retrieval (sentence-transformers + FAISS)

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)

index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

def retrieve_baseline(query, k=3):
    q_emb = model.encode([query])
    _, idx = index.search(q_emb, k)
    return [ids[i] for i in idx[0]]

## Step 2: Hybrid — BM25 keyword search + dense vector, score-normalized and combined

In [ ]:
tokenized_corpus = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized_corpus)

def retrieve_hybrid(query, k=3, alpha=0.5):
    bm25_scores = np.array(bm25.get_scores(query.lower().split()))
    q_emb = model.encode([query])
    # convert L2 distance to a similarity-like score (higher = more similar)
    vec_scores = -np.linalg.norm(doc_embeddings - q_emb, axis=1)

    def norm(x):
        rng = x.max() - x.min()
        return (x - x.min()) / (rng + 1e-9)

    combined = alpha * norm(bm25_scores) + (1 - alpha) * norm(vec_scores)
    top_idx = np.argsort(combined)[::-1][:k]
    return [ids[i] for i in top_idx]

## Step 3: Evaluation — Recall@k and MRR, plus a breakdown by question style

Each test question is tagged `keyword` (reuses exact directive/error-code terms) or `semantic` (paraphrased, no keyword overlap) so you can see *where* each method wins or loses — this breakdown is the core evidence for the paper's Results section.

In [ ]:
def evaluate(retrieve_fn, k=3):
    hits, mrr_total = 0, 0.0
    per_style = {"keyword": {"n": 0, "hits": 0}, "semantic": {"n": 0, "hits": 0}}
    for item in qa:
        preds = retrieve_fn(item['question'], k=k)
        style = item['style']
        per_style[style]['n'] += 1
        if item['expected_doc_id'] in preds:
            hits += 1
            per_style[style]['hits'] += 1
            mrr_total += 1.0 / (preds.index(item['expected_doc_id']) + 1)
    n = len(qa)
    return {
        "recall@k": hits / n,
        "mrr": mrr_total / n,
        "recall_keyword_style": per_style['keyword']['hits'] / per_style['keyword']['n'],
        "recall_semantic_style": per_style['semantic']['hits'] / per_style['semantic']['n'],
    }

for k in [1, 3]:
    print(f"--- k={k} ---")
    print("Baseline (dense vector):", evaluate(retrieve_baseline, k))
    print("Hybrid (BM25 + vector): ", evaluate(retrieve_hybrid, k))

## Step 4: Save results for the paper's Results table

In [ ]:
results = {}
for k in [1, 3]:
    results[f"baseline_k{k}"] = evaluate(retrieve_baseline, k)
    results[f"hybrid_k{k}"]   = evaluate(retrieve_hybrid, k)

with open('/content/results.json', 'w') as f:
    json.dump(results, f, indent=2)

results

## Notes for the paper
- Report Recall@1, Recall@3, and MRR for both methods, plus the keyword-vs-semantic breakdown — that breakdown is your strongest evidence, since it directly shows *why* hybrid helps (exact directive names / error codes) rather than just an aggregate number.
- If hybrid doesn't clearly beat baseline at some k, report that honestly — a null/mixed result with an explanation is still a valid, publishable finding, especially for a short workshop-style paper.
- Try alpha values other than 0.5 (e.g. 0.3, 0.7) as an ablation — cheap to run, gives you an extra results subsection.
- Dataset and questions are entirely synthetic/original text — no Nginx documentation was copied verbatim, so there is no copyright/licensing concern.